In [1]:
import matplotlib.pyplot as plt
import pandas_datareader.data as web
import datetime
import pandas as pd
from dbnomics import fetch_series
from utils import fetch_imf_data, fetch_imf_bulk
from config import country_dict, us_economy_metrics, country_dict_3_char


In [8]:
# Paths
imf_cpi_path = "data/imf_cpi.csv"
imf_gdp_path = "data/imf_gdp.csv"
imf_unemployment_path = "data/imf_unemployment_rate.csv"
imf_industrial_production_index_path = "data/imf_industrial_production_index.csv"
imf_exports_path = "data/imf_exports.csv"
imf_imports_path = "data/imf_imports.csv"
us_economy_path = "data/us_economy.csv"
cb_policy_rates_path = "data/cb_policy_rates.csv"

target_countries = ["VN", "TH", "SG", "ID", "MY", "PH", "BN", "KH", "LA", "MM", "TL", "DE", "FR", "IT", "ES", "NL", "GB", "BE", "AT", "PT", "GR", "FI", "IE", "DK", "SE", "KW", "OM", "BH"]


In [ ]:
# 1. REAL GDP GROWTH (Tăng trưởng GDP Thực tế - Hàng Quý 'Q' hoặc Hàng Năm 'A')
print("Fetching Real GDP Growth Data...")
imf_gdp = fetch_imf_data(
    country_dict=country_dict, 
    dataset_code = "IFS",
    metric_suffix="NGDP_R_XDC",
    frequency="A"
).dropna(subset=['value'])

# Tính YoY % trực tiếp trong Python nếu chuỗi dữ liệu gốc là dạng Số tuyệt đối
imf_gdp['value_yoy'] = imf_gdp.groupby('country_name')['value'].pct_change(periods=4) 
imf_gdp.to_csv(imf_gdp_path, index=False)

Fetching Real GDP Growth Data...


In [5]:
# 2. Unemployment Rate
unemployment_list = []
for country_code in country_dict_3_char.keys():
    df = fetch_series(f"IMF/WEO:2025-04/{country_code}.LUR.pcent_total_labor_force")
    unemployment_list.append(df)
unemployment_raw = pd.concat(unemployment_list, ignore_index=True)
unemployment_raw.to_csv(imf_unemployment_path)

In [ ]:
# 3. INDUSTRIAL PRODUCTION INDEX - IPI (Thay đổi dataset_code="IFS")
ipi_raw = fetch_imf_bulk(
    dataset_code="IFS",
    metric_code="AIP_IX", 
    frequency="M", 
    country_list=target_countries
)
if not ipi_raw.empty:
    ipi_raw.dropna(subset=['value']).to_csv(imf_industrial_production_index_path, index=False)

--- Đang tải dữ liệu cho mã: AIP_IX (M) ---
✅ Tải thành công 11153 dòng dữ liệu!


In [ ]:
# 4. CPI
imf_cpi = fetch_imf_data(country_dict=country_dict, dataset_code = "CPI", metric_suffix="PCPI_IX", frequency="M").dropna(subset=['value'])
# Show 10 samples
imf_cpi.head(10)
# Export data
imf_cpi.to_csv(imf_cpi_path)

In [10]:
# 5. Central bank policy rates
cb_policy_rates_list = []
for country_code in country_dict.keys():
    cb_policy_rates_df = fetch_series(f"BIS/WS_CBPOL/D.{country_code}")
    cb_policy_rates_list.append(cb_policy_rates_df)

cb_policy_rates = pd.concat(cb_policy_rates_list, ignore_index=True)
cb_policy_rates.to_csv(cb_policy_rates_path)

Could not load series: {'dataset_code': 'WS_CBPOL', 'message': 'Could not load series', 'provider_code': 'BIS', 'series_code': 'D.VN'}
Could not load series: {'dataset_code': 'WS_CBPOL', 'message': 'Could not load series', 'provider_code': 'BIS', 'series_code': 'D.SG'}
Could not load series: {'dataset_code': 'WS_CBPOL', 'message': 'Could not load series', 'provider_code': 'BIS', 'series_code': 'D.BN'}
Could not load series: {'dataset_code': 'WS_CBPOL', 'message': 'Could not load series', 'provider_code': 'BIS', 'series_code': 'D.KH'}
Could not load series: {'dataset_code': 'WS_CBPOL', 'message': 'Could not load series', 'provider_code': 'BIS', 'series_code': 'D.LA'}
Could not load series: {'dataset_code': 'WS_CBPOL', 'message': 'Could not load series', 'provider_code': 'BIS', 'series_code': 'D.MM'}
Could not load series: {'dataset_code': 'WS_CBPOL', 'message': 'Could not load series', 'provider_code': 'BIS', 'series_code': 'D.TL'}
Could not load series: {'dataset_code': 'WS_CBPOL', 'me

In [ ]:
# 6. Retail Sales Growth
# DBnomics: OECD (KEI).

In [ ]:
# 7. Global Manufacturing PMI
# DBnomics: WB (World Bank - Pink Sheet Commodity Prices) hoặc IMF.

In [ ]:
# 8. Merchandise Exports/Imports Growth
# DBnomics: WTO (World Trade Organization) hoặc IMF (Bộ dữ liệu DOT - Direction of Trade Statistics).

In [ ]:
# 9. Commodity Price Index
# DBnomics: WB (World Bank - Pink Sheet Commodity Prices) hoặc IMF.

In [7]:
# Timeframe covering Trump baseline through the entirety of the Biden administration
start = datetime.datetime(2000, 1, 1)
end = datetime.datetime(2025, 12, 31)

# Define the exact FRED series codes mapped to your portfolio naming convention
try:
    df_list = []
    for name, code in us_economy_metrics.items():
        print(f"-> Fetching {name} ({code})...")
        # Pull data directly from St. Louis Fed servers
        df = web.DataReader(code, 'fred', start, end)
        # 1. Clean the individual metric's index and rename columns
        df = df.reset_index().rename(columns={'DATE': 'Date', code: 'Value'})
        # 2. Handle frequency alignment: Forward-fill quarterly data before stacking
        # This ensures April and May inherit Q1 data before it gets mixed with monthly metrics
        if name in ['Real_GDP', 'Manufacturing_Investment']:
            # Create a continuous monthly date range to map the quarterly data onto
            monthly_range = pd.date_range(start=start, end=end, freq='MS')
            df = df.set_index('Date').reindex(monthly_range).ffill().reset_index().rename(columns={'index': 'Date'})
        # 3. Add the metadata column so we know which metric this row belongs to
        df['Metric_Name'] = name
        # Reorder columns to look clean: Date | Metric_Name | Value
        df = df[['Date', 'Metric_Name', 'Value']]        
        df_list.append(df)
    print("Consolidating and formatting data frequencies...")
    # Concatenate all tables along the Date axis
    bi_dataset = pd.concat(df_list, axis=0, ignore_index=True)
    # Forward-fill (ffill) the quarterly data (GDP & Investment) so monthly rows aren't blank
    # This prevents relationship errors inside Power BI's model
    bi_dataset = bi_dataset.ffill()
    # Reset index to turn the Date from an index into a normal clean column
    bi_dataset = bi_dataset.reset_index().rename(columns={'DATE': 'Date'})
    # Export to local project directory
    bi_dataset.to_csv(us_economy_path, index=False)
    print(f"Success! Master file saved as '{us_economy_path}'")
    print(bi_dataset.tail(10))

except Exception as e:
    print(f"Pipeline Execution Failed: {e}")

-> Fetching CPI_All_Items (CPIAUCSL)...
-> Fetching Unemployment_Rate (UNRATE)...
-> Fetching Total_Nonfarm_Payrolls (PAYEMS)...
-> Fetching Real_GDP (GDPC1)...
-> Fetching Manufacturing_Investment (C307RX1Q020SBEA)...
Consolidating and formatting data frequencies...
Success! Master file saved as 'data/us_economy.csv'
      index       Date               Metric_Name    Value
1550   1550 2025-03-01  Manufacturing_Investment  145.228
1551   1551 2025-04-01  Manufacturing_Investment  142.245
1552   1552 2025-05-01  Manufacturing_Investment  142.245
1553   1553 2025-06-01  Manufacturing_Investment  142.245
1554   1554 2025-07-01  Manufacturing_Investment  136.654
1555   1555 2025-08-01  Manufacturing_Investment  136.654
1556   1556 2025-09-01  Manufacturing_Investment  136.654
1557   1557 2025-10-01  Manufacturing_Investment  126.551
1558   1558 2025-11-01  Manufacturing_Investment  126.551
1559   1559 2025-12-01  Manufacturing_Investment  126.551
